# 05 - Hidden-feature steering and capstone comparison

The final notebook moves the intervention from 2D sample space into a hidden
layer of the unconditional velocity network.

We collect hidden features from labeled examples after forward noising, learn a
target-vs-rest direction offline, and add that direction during sampling. No
gradient is computed at inference.

**Learning goals**

1. Measure when hidden activations become class-informative.
2. Learn and visualize a class direction in feature space.
3. Test whether one direction transfers across nearby timesteps.
4. Compare baseline, noise alignment, gradient guidance, activation steering,
   and a two-stage combination under paired conditions.


In [ ]:
import sys
from pathlib import Path

COURSE_DIR = Path.cwd()
if not (COURSE_DIR / "toy_course.py").exists():
    COURSE_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(COURSE_DIR.resolve()))

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_course import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print(environment_summary(device))


In [ ]:
model, losses, trained_now = load_or_train_model(device=device)
reference_data, reference_labels = sample_labeled_mixture(30000, device=device)
stats = estimate_gaussian_stats(reference_data, reference_labels)
TARGET_CLASS = 2


## 1. When is class information visible in the hidden layer?

For each time, we corrupt labeled data with the forward process, record one
hidden layer, and fit a small ridge linear probe. High probe accuracy means the
activation contains linearly readable class information. The probe is a
measurement tool; it is not used by the generative model during training.


In [ ]:
probe_rows = []
directions = {}
for t_value in [0.10, 0.25, 0.40, 0.55, 0.70, 0.85]:
    features, labels = collect_forward_activations(model, t_value=t_value, n_per_class=700)
    permutation = torch.randperm(features.shape[0], device=device)
    split = int(0.7 * features.shape[0])
    train_idx, test_idx = permutation[:split], permutation[split:]
    probe = fit_linear_probe(features[train_idx], labels[train_idx])
    accuracy = (probe.predict(features[test_idx]) == labels[test_idx]).float().mean().item()
    direction = discriminant_direction(features[train_idx], labels[train_idx], TARGET_CLASS)
    directions[t_value] = direction
    probe_rows.append({
        "t": t_value,
        "effective noise std": BASE_STD * (1-t_value) / t_value,
        "linear probe accuracy": accuracy,
    })

probe_table = pd.DataFrame(probe_rows)
display(probe_table.round(3))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(probe_table["t"], probe_table["linear probe accuracy"], marker="o", color="#0072B2")
ax.axhline(1 / len(CLASS_NAMES), color="#777777", ls="--", label="chance")
ax.set(xlabel="t (noise -> data)", ylabel="held-out accuracy", ylim=(0.25, 1.02), title="Class information across time")
ax.grid(alpha=0.2)
ax.legend(frameon=False)
plt.show()


In [ ]:
# Visualize the hidden features at one reference time using PCA only for display.
REFERENCE_T = 0.70
features, labels = collect_forward_activations(model, t_value=REFERENCE_T, n_per_class=700)
projected = pca_project(features)

fig, ax = plt.subplots(figsize=(6, 5))
plot_labeled_points(projected, labels, ax=ax, title=f"Hidden activations projected to 2D at t={REFERENCE_T}", alpha=0.35)
ax.legend(frameon=False)
plt.show()

reference_direction = directions[REFERENCE_T]
cosines = []
for t_value, direction in directions.items():
    cosines.append({"t": t_value, "cosine with reference direction": float(torch.dot(direction, reference_direction).cpu())})
cosine_table = pd.DataFrame(cosines)
display(cosine_table.round(3))

# Null control: destroy the feature-label relationship before fitting a direction.
shuffled_labels = labels[torch.randperm(labels.shape[0], device=device)]
shuffled_direction = discriminant_direction(features, shuffled_labels, TARGET_CLASS)

def true_label_score_gap(direction):
    scores = features @ direction
    return float((scores[labels == TARGET_CLASS].mean() - scores[labels != TARGET_CLASS].mean()).cpu())

null_control = pd.DataFrame({
    "direction": ["true labels", "shuffled labels"],
    "target-vs-rest score gap": [
        true_label_score_gap(reference_direction),
        true_label_score_gap(shuffled_direction),
    ],
})
display(null_control.round(3))
assert null_control.loc[0, "target-vs-rest score gap"] > 0


## 2. Activation intervention

The target direction is covariance-aware: it separates target examples from
the remaining classes in hidden space. During an active time window, the model
adds a scaled unit direction to the selected hidden layer. The scale is relative
to the current feature RMS, so it remains comparable across time.

This is a genuine hidden-feature intervention, not sample-space attraction. It
is still simpler than NA-RFM: the paper uses RFM directions in image-model
activation tensors, while this notebook uses a linear discriminant direction in
one MLP layer.


In [ ]:
set_seed(SEED + 50)
x0 = sample_base(1400, device=device)
target_reference = sample_class(1400, TARGET_CLASS, device=device)

methods = {
    "baseline": base_velocity(model),
    "noise alignment": make_noise_aligned_velocity(
        model, stats, target_class=TARGET_CLASS, strength=1.6
    ),
    "gradient guidance": make_gradient_guided_velocity(
        model, stats, target_class=TARGET_CLASS, strength=3.0, start=0.15, end=0.92
    ),
    "activation steering": make_activation_steered_velocity(
        model, reference_direction, strength=4.0, start=0.38, end=0.90
    ),
    "noise + activation": make_combined_velocity(
        model, stats, reference_direction,
        target_class=TARGET_CLASS, noise_strength=1.6, activation_strength=4.0,
    ),
}

rows = []
paths = {}
for name, velocity in methods.items():
    path, seconds = timed_sample(velocity, x0, steps=56, method="heun")
    metrics = evaluate_samples(path[-1].to(device), target_reference, stats, target_class=TARGET_CLASS)
    rows.append({"method": name, **metrics, "seconds": seconds})
    paths[name] = path

capstone_table = pd.DataFrame(rows)
capstone_table["runtime / baseline"] = capstone_table["seconds"] / capstone_table.loc[0, "seconds"]
display(capstone_table.round(4))


In [ ]:
plot_trajectory_comparison(
    {
        "baseline": paths["baseline"],
        "noise alignment": paths["noise alignment"],
        "activation steering": paths["activation steering"],
        "noise + activation": paths["noise + activation"],
    },
    target_class=TARGET_CLASS,
    n_lines=18,
)
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 5))
for _, row in capstone_table.iterrows():
    ax.scatter(row["target_swd"], row["target_rate"], s=80)
    ax.annotate(row["method"], (row["target_swd"], row["target_rate"]), xytext=(5, 4), textcoords="offset points", fontsize=9)
ax.set(xlabel="SWD to target class (lower is better)", ylabel="target-class rate (higher is better)", title="Control-quality trade-off")
ax.grid(alpha=0.2)
plt.show()


## 3. Capstone ablation

Choose one factor at a time and repeat the paired protocol:

1. **Collection time:** learn directions at `t = 0.40, 0.55, 0.70, 0.85`.
2. **Intervention window:** early, middle, or late with fixed direction/strength.
3. **Activation strength:** at least three values with fixed direction/window.
4. **Method:** baseline, noise alignment, gradient, activation, and combined.
5. **Robustness:** repeat the decisive comparison over three initial-noise seeds.

Report target-class rate, target SWD, diversity ratio, runtime, and one paired
trajectory figure. A method is not "best" unless the quality constraint used to
select it is stated.

## Paper bridge

| NA-RFM component | Toy implementation | Important limitation |
|---|---|---|
| High-noise noise alignment | class Gaussian denoiser minus full Gaussian denoiser | 2D full covariance, not image PCA |
| Forward activation collection | labeled points corrupted to a reference `t` | one small MLP layer |
| Target direction | covariance-aware linear discriminant | not Recursive Feature Machine |
| Online activation edit | fixed feature direction over a time window | no U-Net feature map or amplification pass |
| Gradient-free inference | yes for noise/activation methods | gradient baseline still backpropagates |

**Final questions**

1. Does probe accuracy predict steering success at the same collection time?
2. Over which times is the learned direction stable enough to reuse?
3. When does combining coarse high-noise and later activation control help?
4. Which conclusion from 2D would require a new experiment before claiming it for images?
